### 8-Puzzle Problem Solver using A* Search Algorithm

This notebook implements the A* search algorithm to find the optimal path to solve an 8-puzzle board.
We use the **Manhattan Distance** as our heuristic.

In [1]:
import heapq
import copy

class PuzzleNode:
    def __init__(self, state, parent=None, action=None, g=0, h=0):
        self.state = state  # 2D tuple (3x3)
        self.parent = parent
        self.action = action
        self.g = g  # Path cost from start node
        self.h = h  # Heuristic cost to goal node
        self.f = g + h  # Total cost

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.state[r][c] == 0:
                    return r, c

    def get_neighbors(self):
        neighbors = []
        r, c = self.get_blank_pos()
        moves = {
            'UP': (r - 1, c),
            'DOWN': (r + 1, c),
            'LEFT': (r, c - 1),
            'RIGHT': (r, c + 1)
        }

        for action, (nr, nc) in moves.items():
            if 0 <= nr < 3 and 0 <= nc < 3:
                # Create new state
                new_state = [list(row) for row in self.state]
                # Swap blank (0) with the target cell
                new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
                # Convert back to tuple for hashability
                tuple_state = tuple(tuple(row) for row in new_state)
                neighbors.append((tuple_state, action))
        return neighbors

In [2]:
def manhattan_distance(state, goal_state):
    distance = 0
    # Precompute goal positions
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal_state[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                gr, gc = goal_pos[val]
                distance += abs(r - gr) + abs(c - gc)
    return distance

In [3]:
def solve_a_star(start_state, goal_state):
    start_h = manhattan_distance(start_state, goal_state)
    start_node = PuzzleNode(start_state, g=0, h=start_h)

    open_set = []
    heapq.heappush(open_set, start_node)

    closed_set = set()

    nodes_explored = 0

    while open_set:
        current_node = heapq.heappop(open_set)
        nodes_explored += 1

        if current_node.state == goal_state:
            path = []
            curr = current_node
            while curr:
                path.append((curr.state, curr.action))
                curr = curr.parent
            return path[::-1], nodes_explored

        closed_set.add(current_node.state)

        for neighbor_state, action in current_node.get_neighbors():
            if neighbor_state in closed_set:
                continue

            g_score = current_node.g + 1
            h_score = manhattan_distance(neighbor_state, goal_state)
            neighbor_node = PuzzleNode(neighbor_state, parent=current_node, action=action, g=g_score, h=h_score)

            # Check if neighbor is already in open_set with a lower/equal f score
            # Simplified check using heapq properties
            heapq.heappush(open_set, neighbor_node)

    return None, nodes_explored

In [4]:
def print_board(state):
    for row in state:
        print(" ".join(str(x) if x != 0 else "_" for x in row))
    print()

# Define Start and Goal States
# 0 represents the blank space
start_state = (
    (1, 2, 3),
    (0, 4, 6),
    (7, 5, 8)
)

goal_state = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

print("Start State:")
print_board(start_state)
print("Goal State:")
print_board(goal_state)

path, explored = solve_a_star(start_state, goal_state)

if path:
    print(f"Solved in {len(path) - 1} moves. Explored {explored} nodes.\n")
    for step, (state, action) in enumerate(path):
        if action:
            print(f"Step {step}: Move {action}")
        else:
            print("Initial State:")
        print_board(state)
else:
    print("No solution found.")

Start State:
1 2 3
_ 4 6
7 5 8

Goal State:
1 2 3
4 5 6
7 8 _

Solved in 3 moves. Explored 4 nodes.

Initial State:
1 2 3
_ 4 6
7 5 8

Step 1: Move RIGHT
1 2 3
4 _ 6
7 5 8

Step 2: Move DOWN
1 2 3
4 5 6
7 _ 8

Step 3: Move RIGHT
1 2 3
4 5 6
7 8 _

